In [83]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Jour 1 — Phase 1 : Charger et explorer un dataset de classification

## Dataset utilisé
- `load_breast_cancer`
- 569 observations
- 30 variables explicatives
- 2 classes : bénigne / maligne

## Objectif
Créer une fonction `explorer_dataset()` qui :
- charge le dataset,
- affiche le nombre de lignes et de colonnes,
- affiche les classes possibles,
- affiche le nombre d’exemples par classe,
- affiche la répartition en pourcentage,
- détecte rapidement si le dataset est équilibré ou non.

In [84]:
from sklearn.datasets import load_breast_cancer
import numpy as np

In [85]:
def explorer_dataset(X=None, y=None, target_names=None):
    """
    Explore un dataset de classification et affiche :
    - le nombre de lignes / colonnes
    - les classes possibles
    - le nombre d'exemples par classe
    - la répartition en pourcentage
    - un diagnostic simple sur l'équilibre du dataset

    Si X, y et target_names ne sont pas fournis,
    le dataset breast cancer est chargé automatiquement.
    """

    # 1) Chargement automatique du dataset si aucun argument n'est fourni
    if X is None or y is None:
        data = load_breast_cancer()
        X = data.data
        y = data.target
        target_names = data.target_names

    # 2) Forme du dataset
    print("=== Informations générales ===")
    print(f"Lignes, colonnes : {X.shape}")

    # 3) Classes et effectifs
    classes, counts = np.unique(y, return_counts=True)
    total = len(y)

    print("\n=== Répartition des classes ===")
    for classe, count in zip(classes, counts):
        if target_names is not None and len(target_names) > classe:
            nom_classe = target_names[classe]
        else:
            nom_classe = f"Classe {classe}"

        pourcentage = (count / total) * 100
        print(f"{nom_classe} : {count} cas ({pourcentage:.2f}%)")

    # 4) Diagnostic très simple sur l'équilibre
    ratio_majoritaire = counts.max() / total

    print("\n=== Diagnostic rapide ===")
    if ratio_majoritaire >= 0.90:
        print("Dataset très déséquilibré : une classe domine très fortement.")
    elif ratio_majoritaire >= 0.70:
        print("Dataset modérément déséquilibré.")
    else:
        print("Dataset plutôt équilibré.")

In [86]:
print("=== TEST 1 : dataset complet ===")
explorer_dataset()

=== TEST 1 : dataset complet ===
=== Informations générales ===
Lignes, colonnes : (569, 30)

=== Répartition des classes ===
malignant : 212 cas (37.26%)
benign : 357 cas (62.74%)

=== Diagnostic rapide ===
Dataset plutôt équilibré.


In [87]:
data = load_breast_cancer()
X = data.data
y = data.target
target_names = data.target_names

In [88]:
# On garde uniquement la classe 0
mask = (y == 0)
X_single = X[mask]
y_single = y[mask]

print("=== TEST 2 : une seule classe (y == 0) ===")
explorer_dataset(X_single, y_single, target_names)

=== TEST 2 : une seule classe (y == 0) ===
=== Informations générales ===
Lignes, colonnes : (212, 30)

=== Répartition des classes ===
malignant : 212 cas (100.00%)

=== Diagnostic rapide ===
Dataset très déséquilibré : une classe domine très fortement.


In [89]:
print("=== TEST 3 : vérification de la répartition en pourcentage ===")
explorer_dataset()

=== TEST 3 : vérification de la répartition en pourcentage ===
=== Informations générales ===
Lignes, colonnes : (569, 30)

=== Répartition des classes ===
malignant : 212 cas (37.26%)
benign : 357 cas (62.74%)

=== Diagnostic rapide ===
Dataset plutôt équilibré.


## Conclusion

La fonction `explorer_dataset()` fonctionne sur :
- le dataset complet,
- un cas limite avec une seule classe,
- un affichage en pourcentage pour rendre visible un éventuel déséquilibre.

La phase 1 est terminée.
La suite logique est la phase 2 :
- split train/test,
- entraînement,
- prédiction,
- évaluation avec l'accuracy.

# Phase 2 — Pipeline supervisé complet

Dans cette phase, on met en place un pipeline de Machine Learning complet :

- séparation des données (train / test)
- entraînement d’un modèle
- prédiction
- évaluation avec l’accuracy

Le tout sera encapsulé dans une fonction réutilisable.

In [90]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [91]:
# Chargement du dataset
data = load_breast_cancer()
X = data.data
y = data.target

In [92]:
# Séparation train / test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("=== Split des données ===")
print("Train :", X_train.shape)
print("Test  :", X_test.shape)

=== Split des données ===
Train : (455, 30)
Test  : (114, 30)


## Fonction d'entraînement et d'évaluation

On crée une fonction générique qui :
- prend un modèle en entrée
- l'entraîne
- fait des prédictions
- calcule l'accuracy

In [93]:
def entrainer_et_evaluer(modele, X_train, X_test, y_train, y_test):
    """
    Entraîne un modèle, prédit sur le test et retourne l'accuracy.
    """

    # 1) Entraînement
    modele.fit(X_train, y_train)

    # 2) Prédiction
    y_pred = modele.predict(X_test)

    # 3) Évaluation
    accuracy = accuracy_score(y_test, y_pred)

    return accuracy

## Test du modèle

On teste le pipeline avec un Decision Tree.

In [94]:
# Création du modèle
modele = DecisionTreeClassifier(random_state=42)

# Évaluation
accuracy = entrainer_et_evaluer(
    modele,
    X_train,
    X_test,
    y_train,
    y_test
)

print("=== Résultat du modèle ===")
print(f"Accuracy : {accuracy:.2%}")

=== Résultat du modèle ===
Accuracy : 91.23%


## Conclusion Phase 2

Le pipeline Machine Learning fonctionne correctement :

- les données sont séparées (train/test)
- le modèle est entraîné
- des prédictions sont réalisées
- la performance est mesurée avec l'accuracy

La fonction `entrainer_et_evaluer()` permet de tester facilement d'autres modèles.

Prochaine étape : comparer plusieurs algorithmes (Phase 3).

# Phase 3 — L'Arène des algorithmes

Dans cette phase, on compare plusieurs modèles de Machine Learning sur le même dataset.

Objectifs :
- tester plusieurs algorithmes
- utiliser le même split train/test
- comparer leurs performances (accuracy)
- afficher un classement

In [95]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

## Fonction Arène

Cette fonction va :
- entraîner plusieurs modèles
- calculer leur accuracy
- trier les résultats
- afficher un classement

In [96]:
def arene(X_train, X_test, y_train, y_test):
    """
    Entraîne plusieurs modèles et affiche un classement trié par accuracy.
    """

    # 1) Définition des modèles
    modeles = {
        "Régression Logistique": LogisticRegression(max_iter=200),
        "KNN (k=3)": KNeighborsClassifier(n_neighbors=3),
        "Arbre de décision": DecisionTreeClassifier(random_state=42),
    }

    resultats = []

    # 2) Boucle sur les modèles
    for nom, modele in modeles.items():
        accuracy = entrainer_et_evaluer(
            modele,
            X_train,
            X_test,
            y_train,
            y_test
        )
        resultats.append((nom, accuracy))

    # 3) Tri par accuracy décroissante
    resultats = sorted(resultats, key=lambda x: x[1], reverse=True)

    # 4) Affichage
    print("=== Classement des modèles ===\n")
    for i, (nom, score) in enumerate(resultats, start=1):
        print(f"{i}. {nom} : {score:.2%}")

    return resultats

## Lancement de l'Arène

In [97]:
resultats = arene(X_train, X_test, y_train, y_test)

=== Classement des modèles ===

1. Régression Logistique : 96.49%
2. KNN (k=3) : 92.98%
3. Arbre de décision : 91.23%


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Analyse des résultats

On observe que :
- certains modèles performent mieux que d'autres
- la régression logistique donne souvent de très bons résultats sur ce dataset
- le KNN est performant mais dépend des distances
- l'arbre de décision est simple mais parfois moins précis

Conclusion :
Le choix d’un modèle dépend des performances mais aussi de la stabilité et de l'interprétabilité.

In [98]:
modeles_test = {
    "KNN (k=3)": KNeighborsClassifier(n_neighbors=3),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "KNN (k=10)": KNeighborsClassifier(n_neighbors=10),
}

print("=== Test variation de KNN ===\n")

for nom, modele in modeles_test.items():
    accuracy = entrainer_et_evaluer(
        modele,
        X_train,
        X_test,
        y_train,
        y_test
    )
    print(f"{nom} : {accuracy:.2%}")

=== Test variation de KNN ===

KNN (k=3) : 92.98%
KNN (k=5) : 91.23%
KNN (k=10) : 92.98%


# Phase 4 — Clustering (apprentissage non supervisé)

Dans cette phase, on utilise un algorithme de clustering (KMeans) :

- les labels sont cachés
- le modèle doit trouver des groupes tout seul
- on compare ensuite avec les vraies classes

Objectif :
Voir si la structure des données permet de retrouver les classes sans supervision.

In [99]:
from sklearn.cluster import KMeans

## Fonction de clustering

Le modèle :
- ne voit pas les labels
- regroupe les données en 2 clusters

In [100]:
def clustering_aveugle(X):
    """
    Applique KMeans sans les labels.
    Retourne les clusters trouvés.
    """

    kmeans = KMeans(n_clusters=2, random_state=42)
    clusters = kmeans.fit_predict(X)

    return clusters

## Application du clustering

In [101]:
# Import des données
data = load_breast_cancer()
X = data.data
y = data.target

# Clustering sans labels
clusters = clustering_aveugle(X)

# Aperçu
print("Clusters trouvés (aperçu) :", clusters[:20])

Clusters trouvés (aperçu) : [0 0 0 1 0 1 0 1 1 1 1 0 0 1 1 1 1 0 0 1]


## Comparaison avec les vraies classes

In [102]:
from sklearn.metrics import accuracy_score

# ATTENTION : les labels peuvent être inversés
accuracy_1 = accuracy_score(y, clusters)
accuracy_2 = accuracy_score(y, 1 - clusters)

accuracy = max(accuracy_1, accuracy_2)

print("=== Comparaison clustering vs réalité ===")
print(f"Accuracy approximative : {accuracy:.2%}")


=== Comparaison clustering vs réalité ===
Accuracy approximative : 85.41%


## Analyse

Le clustering KMeans permet de retrouver approximativement les classes réelles.

Cependant :
- le modèle ne connaît pas les labels
- il se base uniquement sur la structure des données

Résultat :
- si l’accuracy est élevée → la structure est naturellement séparée
- sinon → les classes ne sont pas facilement séparables

Conclusion :
Le clustering peut parfois retrouver la réalité, mais ce n’est pas garanti.

In [ ]:
import numpy as np

unique, counts = np.unique(clusters, return_counts=True)

print("Répartition des clusters :")
for u, c in zip(unique, counts):
    print(f"Cluster {u} : {c} points")